In [ ]:
# --- Colab bootstrap: in Colab, clone + install the repo; locally, do nothing ---
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/CKmellow/per3-genotype-clock-gene-timing.git"
BRANCH = "main"  # change to your branch name to test it in Colab
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    repo = Path("/content/per3-genotype-clock-gene-timing")
    if not repo.exists():
        subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, REPO_URL, str(repo)], check=True)
    os.chdir(repo)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=True)
    src = str(repo / "src")
else:  # local: find the repo root (works from notebooks/ or notebooks/stages/)
    here = Path.cwd().resolve()
    root = next((p for p in [here, *here.parents] if (p / "pyproject.toml").exists()), here)
    src = str(root / "src")
if src not in sys.path:
    sys.path.insert(0, src)  # makes `import per3clock` work without a kernel restart

from per3clock import config  # noqa: E402

config.set_seeds()
config.ensure_dirs()
print("per3clock root:", config.ROOT, "| Colab:", IN_COLAB)

# Stage 7 · Machine learning: predicting internal circadian phase

*Owner: Part 5 (issue #5).*

## Minimum evidence (lecturer's brief)
*Brief, stage 7: "Build a model that classifies, predicts or groups your data." Learning outcome 6: "Build and evaluate machine learning models against proper baselines."*

- [x] A model that **predicts** each sample's internal circadian phase (`circadian_phase_h`) from expression
- [x] **Evaluated against proper baselines** with leave-one-subject-out CV (no leakage: scaling and feature selection inside folds)
- [x] Metric: **median absolute circular error in hours**, with the same folds for every model
- [x] A fairer "added value" comparison: **clock time + genes vs clock time alone**
- [x] Short interpretation of your own results
- [x] Report what the data does **and does not** show. A null, negative or inconclusive result is a valid result: report it with its uncertainty. Matching the clock-time baseline without beating it is an acceptable, reportable result.

## Contract
**Samples:** `data.analysis_samples(for_phase_analysis=True)` (target is `circadian_phase_h`; AF0182 SE dropped; 23 subjects, 214 samples). **Features:** clock genes via `data.load_clock_genes()` (one probe per gene) and/or genes from `data.collapse_probes_to_genes(data.load_expression_matrix())`.

**Target:** `circadian_phase_h` encoded as sin/cos. **CV:** `LeaveOneGroupOut`/`GroupKFold` by `subject`. **Metric:** median absolute circular error (h).
**Baselines:** (a) circular mean phase of the training fold; (b) phase predicted from `clock_time_h` only. Optional: add PER3 genotype as a feature.

**Read this before interpreting (data caveat):** `circadian_phase_h` comes in **3-h steps** and is roughly **clock time minus a per-subject offset** (the clock time of each person's melatonin phase 0, ~02:15–07:25 in SE). So baseline (b) will be **strong**: it is only wrong by that offset. Compare the model with it honestly, using the same LOSO folds and the median absolute circular error. **Matching but not beating the clock-time baseline is an acceptable, reportable result.** The fairer question is whether expression adds information beyond clock time: compare **clock time + genes** against **clock time alone** (e.g. does adding genes reduce the per-subject offset error?). Errors below ~1.5 h are not meaningful with 3-h bins. GSE48113 has no phase labels, so there is no external validation set (`data/README.md`).

**Implement in `per3clock.ml`:** `encode_phase`, `decode_phase`, `circular_abs_error_h`, `loso_cv_predict`, `evaluate_phase_models`.

**Outputs:** `results/tables/ml_phase_prediction.csv` (one row per model, including both baselines and clock time + genes); figures `ml_predicted_vs_true_phase.png`, `ml_error_by_model.png`.

In [ ]:
import numpy as np
import pandas as pd
from IPython.display import display

from per3clock import config, data, ml, plotting

plotting.set_style()
pd.set_option("display.width", 160)

## 7.1 Features and target

**Samples:** `analysis_samples(for_phase_analysis=True)`: Sleep Extension, 23 subjects, 214 samples (AF0182 dropped because its phase labels are inconsistent).
**Target:** `circadian_phase_h` (0 ≈ melatonin midpoint), wrapped to 0–24 h and encoded as (sin, cos) so that 23 h and 1 h are 2 h apart, not 22 h.
**Groups:** subject. Every model is scored with **leave-one-subject-out (LOSO)** CV: train on 22 people, predict all samples of the 23rd. So a score measures prediction for a *new person*.

**Three feature sets, fixed before any model was scored:**
1. **Clock genes:** the 8 clock genes (one probe per gene, the shared default).
2. **Genome-wide:** all 17,528 expressed genes from Stage 3. Inside each training fold, keep the **50 genes** whose expression is best explained by a cosinor fit on the training subjects' phase (`ml.TopRhythmicFeatures`), then scale and fit ridge regression. The test subject is never used for selection, scaling or the ridge penalty (`RidgeCV`, also inside the fold). k = 50 was chosen in advance; k = 20 and 200 are a sensitivity check (7.5).
3. **Genome-wide, within-subject centred:** the same, after subtracting each person's own mean expression from their samples. Stage 4 showed that the *person* explains ~77 % of expression variation and time of day only ~4 %; centring removes that person effect. It uses **no labels**, only the test subject's own expression, but it needs that person's whole sampled series (10 draws over ~30 h), not a single blood draw.

In [ ]:
meta = data.analysis_samples(for_phase_analysis=True).reset_index(drop=True)
groups = meta["subject"].to_numpy()
phase = np.mod(meta["circadian_phase_h"].to_numpy(), 24)

clock = data.load_clock_genes()
X_clock = (clock.pivot(index="sample_id", columns="gene", values="expression")
           .loc[meta["sample_id"], list(config.CLOCK_GENES)].reset_index(drop=True))

expressed = pd.read_csv(config.PROCESSED_DIR / "expressed_genes_SE.csv")["gene"].tolist()
genes = data.collapse_probes_to_genes(data.load_expression_matrix())
X_genome = genes.loc[expressed, meta["sample_id"]].T.reset_index(drop=True)
X_centred = ml.within_subject_centre(X_genome, groups)

print(f"{meta.subject.nunique()} subjects ({meta.drop_duplicates('subject').per3_genotype.value_counts().to_dict()}), "
      f"{len(meta)} samples")
print(f"clock genes: {X_clock.shape[1]} | genome-wide: {X_genome.shape[1]:,} expressed genes")
print("phase values (h, mod 24):", sorted(set(phase)))

## 7.2 Baselines

- **(a) Circular mean phase:** every sample gets the training fold's mean phase. Any useful model must beat this.
- **(b) Clock time only:** phase = clock time − the training fold's mean offset between clock time and phase. In this protocol, phase is clock time minus each person's own offset (when their melatonin peaked), so this baseline is wrong only by how far a person's offset differs from the group's. **It is strong by design.**

In [ ]:
K = 50  # genes selected inside each fold; fixed in advance
models = {}
models.update(ml.default_phase_models(X_clock, meta, label="clock genes"))
gw = ml.default_phase_models(X_genome, meta, k=K, label=f"genome top-{K}", with_genotype=False)
gc = ml.default_phase_models(X_centred, meta, k=K, label=f"centred genome top-{K}",
                             with_genotype=False)
models.update({k: v for k, v in {**gw, **gc}.items() if not k.startswith("baseline")})
print("\n".join(models))

## 7.3 Models

All non-baseline models are `StandardScaler → RidgeCV` pipelines predicting (sin, cos) of phase, decoded back to hours with `arctan2`. The genome-wide pipelines put the in-fold gene selection first. Ridge is used because there are few samples (214) and correlated genes; it is linear, so its coefficients can be read. We report the **median absolute circular error** with a 95 % CI that resamples **subjects** (samples from one person are not independent).

In [ ]:
pred = ml.phase_predictions(models, meta)
summary = ml.summarise_predictions(pred)
feature_set = {"baseline": "none / clock time", "clock genes": "8 clock genes",
               "centred genome": f"top-{K} of 17,528 genes, centred", "genome": f"top-{K} of 17,528 genes"}
summary.insert(1, "features", summary["model"].map(
    lambda m: next(v for k, v in feature_set.items() if k in m)))
summary.to_csv(config.TABLES_DIR / "ml_phase_prediction.csv", index=False)
display(summary.round(2))

## 7.3b Does expression add information beyond clock time?

Same LOSO folds, paired across the 23 subjects: for each person, take the median error of **clock time + genes** and subtract the median error of **clock time only**. Negative = the genes helped. We report the mean difference with a subject-bootstrap 95 % CI, the Wilcoxon signed-rank p-value, and how many people improved.

In [ ]:
BASE = "baseline: clock time only"
pairs = [(BASE, "clock time + clock genes"),
         (BASE, f"clock time + genome top-{K}"),
         (BASE, f"clock time + centred genome top-{K}"),
         (BASE, f"centred genome top-{K}"),
         ("clock genes", "clock genes + PER3 genotype")]
paired = pd.DataFrame([ml.paired_subject_comparison(pred, a, b) for a, b in pairs])
paired.to_csv(config.TABLES_DIR / "ml_paired_comparisons.csv", index=False)
display(paired.round(3))

## 7.4 Comparison

In [ ]:
fig = ml.plot_predicted_vs_true(pred, ["baseline: clock time only", "clock genes",
                                       f"centred genome top-{K}",
                                       f"clock time + centred genome top-{K}"])
plotting.save_fig(fig, "ml_predicted_vs_true_phase")

In [ ]:
fig = ml.plot_error_by_model(summary, pred, baseline=BASE)
plotting.save_fig(fig, "ml_error_by_model")

**Which genes carry the signal?** Genes picked by the in-fold selection (centred genome-wide model) in how many of the 23 folds. Genes picked in every fold are stable predictors, not artefacts of one split. The column `clock_gene` flags the core clock genes.

In [ ]:
sel = ml.fold_selection_frequency(X_centred, phase, groups, k=K)
scan = pd.read_csv(config.TABLES_DIR / "rhythmic_genes_SE.csv", index_col="gene")
sel = sel.join(scan[["acrophase_h", "amplitude", "q"]], on="gene")
sel["clock_gene"] = sel["gene"].isin(config.CLOCK_GENES + ("NR1D2", "DBP", "TEF", "HLF", "BHLHE40", "BHLHE41", "CRY2", "ARNTL2", "NPAS2", "CIART"))
sel.to_csv(config.TABLES_DIR / "ml_selected_genes.csv", index=False)
print(f"{(sel.frac_folds == 1).sum()} genes selected in all {len(set(groups))} folds")
display(sel.head(20).round(3))
print("clock genes among the selected:", sel.loc[sel.clock_gene, ["gene", "n_folds_selected"]].values.tolist())

## 7.5 Sensitivity: number of selected genes

k = 50 was fixed in advance. Here k = 20 and 200 for the centred genome-wide models, same folds.

In [ ]:
sens = []
for k in (20, 50, 200):
    m = ml.default_phase_models(X_centred, meta, k=k, label=f"centred genome top-{k}",
                                with_genotype=False)
    m = {n: v for n, v in m.items() if not n.startswith("baseline")}
    p = ml.phase_predictions(m, meta)
    s = ml.summarise_predictions(p).assign(k=k)
    c = ml.paired_subject_comparison(pd.concat([p, pred[pred.model == BASE]]), BASE,
                                     f"clock time + centred genome top-{k}")
    s.loc[s.model.str.startswith("clock time"), ["paired_mean_diff_vs_clock_h", "paired_ci95_low_h",
          "paired_ci95_high_h", "paired_wilcoxon_p"]] = [c["mean_diff_h"], c["ci95_low_h"],
                                                         c["ci95_high_h"], c["wilcoxon_p"]]
    sens.append(s)
sens = pd.concat(sens, ignore_index=True)
sens.to_csv(config.TABLES_DIR / "ml_sensitivity_k.csv", index=False)
display(sens.drop(columns=["n_samples", "n_subjects"]).round(3))

## Interpretation

**Does expression predict internal phase?** Yes. Under leave-one-subject-out CV, every expression model beats the "average phase" baseline (4.6 h median error). The 8 clock genes alone reach 2.5 h [95 % CI 2.3–3.0]. The best expression-only model (top-50 genes, centred within each person) reaches 1.1 h [0.9–1.2], with 89 % of samples within 3 h.

**Does it beat clock time?** Expression alone does **not** beat the clock-time baseline (1.1 h [0.9–1.4]). The paired difference is +0.09 h [−0.27, +0.47], p = 0.66. That baseline is strong because this protocol ties phase to clock time.

**Does expression add information beyond clock time?** A little, and consistently. On the same folds, adding the centred genome-wide genes to clock time lowers the error from 1.08 to **0.77 h** [0.55–0.93]. The paired mean is −0.40 h per subject [−0.55, −0.24], Wilcoxon p = 0.0001, and 18 of 23 people improved. The gain holds for k = 20 (−0.30 h) and k = 200 (−0.44 h). Adding the 8 clock genes gives a small gain (−0.08 h [−0.14, −0.02], p = 0.03). Adding *uncentred* genome-wide genes gives none (p = 0.27). Expression only helps once each person's own baseline is removed, which matches Stage 4, where the person dominated expression variation.

**Which genes carry the signal?** 38 genes were selected in all 23 folds. They fall into the two Stage 6 clusters: 22 peak in the biological evening/night (e.g. *LEF1*, *TCF7*, *MAL*, *LRRN3*, *NELL2*: naive T-cell markers) and 15 in the biological day (e.g. *DDIT4*, a glucocorticoid-response gene, and *DAAM2*). *NR1D1* is the only core clock gene picked in every fold. The phase signal in blood therefore comes mostly from **daily changes in white-cell composition and cortisol response**, with a smaller contribution from the intracellular clock.

**Does PER3 genotype help?** No. Adding genotype to the clock-gene model changes the error by −0.01 h [−0.23, +0.16], p = 0.69. This agrees with Stage 5: genotype carries no detectable information about clock-gene timing relative to internal phase.

**What the data do not show.**
- The phase labels come in **3-h steps**, so a gain of 0.4 h is consistent across people but smaller than the label resolution. It shows that the model ranks phase a bit better than clock time does, not that it reaches sub-hour accuracy.
- In Sleep Extension, `circadian_phase_h` equals hours awake − 19.5 for every subject in this set (data finding, Stage 2/5). The target is therefore also **time since waking**, so the model cannot separate the internal clock from a wake-driven ("hours awake") signal.
- Centring needs each person's full 10-sample series, so this is **not** a single-blood-draw phase test.
- There is no external validation set (GSE48113 has no phase labels), so the error is estimated on this one cohort (23 people) only.